In [1]:
import polars as pl
import polars.selectors as cs

path = './Files/Sample_Superstore.csv'
df = pl.read_csv(path)

In [2]:
df1= pl.DataFrame(
    [
        {"year": 2023, "exporter": "India", "importer": "Russia", "quantity": 0 },
        {"year": 2023, "exporter": "India", "importer": "Russia", "quantity": 1 }
    ]
)
df1

year,exporter,importer,quantity
i64,str,str,i64
2023,"""India""","""Russia""",0
2023,"""India""","""Russia""",1


In [3]:
df2 = pl.DataFrame(
    [
        {"year": 2024, "exporter": "India", "importer": "Russia", "quantity": 2 },
        {"year": 2024, "exporter": "India", "importer": "Russia", "quantity": 2 }
    ]
)
df2

year,exporter,importer,quantity
i64,str,str,i64
2024,"""India""","""Russia""",2
2024,"""India""","""Russia""",2


In [4]:
#Vertical combination combine both into one dataframe
df1.vstack(df2) #keeps prevois dataframes and points new dataframe to these locations
#vstack references memory making other operations slower like group by

year,exporter,importer,quantity
i64,str,str,i64
2023,"""India""","""Russia""",0
2023,"""India""","""Russia""",1
2024,"""India""","""Russia""",2
2024,"""India""","""Russia""",2


In [5]:
#REchunk copies data into new location

df1.vstack(df2).rechunk()

year,exporter,importer,quantity
i64,str,str,i64
2023,"""India""","""Russia""",0
2023,"""India""","""Russia""",1
2024,"""India""","""Russia""",2
2024,"""India""","""Russia""",2


In [6]:
#Appends dataframe modifies df1
df1.extend(df2)

year,exporter,importer,quantity
i64,str,str,i64
2023,"""India""","""Russia""",0
2023,"""India""","""Russia""",1
2024,"""India""","""Russia""",2
2024,"""India""","""Russia""",2


In [7]:
df1

year,exporter,importer,quantity
i64,str,str,i64
2023,"""India""","""Russia""",0
2023,"""India""","""Russia""",1
2024,"""India""","""Russia""",2
2024,"""India""","""Russia""",2


In [26]:
df1= pl.DataFrame(
    [
        {"year": 2023, "exporter": "India", "importer": "Russia", "quantity": 0 },
        {"year": 2023, "exporter": "India", "importer": "Russia", "quantity": 1 }
    ]
)

In [10]:
pl.concat((df1, df2), how='vertical') #default= vertical  vstack then rechunk  rechunk=False to just vstack

year,exporter,importer,quantity
i64,str,str,i64
2023,"""India""","""Russia""",0
2023,"""India""","""Russia""",1
2024,"""India""","""Russia""",2
2024,"""India""","""Russia""",2


In [12]:
#For vertical concal df must have same types and names
df2f = df2.with_columns(pl.col('quantity').cast(pl.Float64))
df2f

year,exporter,importer,quantity
i64,str,str,f64
2024,"""India""","""Russia""",2.0
2024,"""India""","""Russia""",2.0


In [15]:
#df1.vstack(df2f) Error
pl.concat(
    (df1, df2f.with_columns(pl.col('quantity').cast(pl.Int64))
))

year,exporter,importer,quantity
i64,str,str,i64
2023,"""India""","""Russia""",0
2023,"""India""","""Russia""",1
2024,"""India""","""Russia""",2
2024,"""India""","""Russia""",2


In [16]:
pl.concat((df1, df2), how='vertical_relaxed')

year,exporter,importer,quantity
i64,str,str,i64
2023,"""India""","""Russia""",0
2023,"""India""","""Russia""",1
2024,"""India""","""Russia""",2
2024,"""India""","""Russia""",2


In [19]:
diffdf = pl.DataFrame(
    [
    {"item": "Clothes", "value": 10},
    {"item": "Machinery", "value": 1000}
    ]
)
diffdf

item,value
str,i64
"""Clothes""",10
"""Machinery""",1000


In [20]:
df1.hstack(diffdf)

year,exporter,importer,quantity,item,value
i64,str,str,i64,str,i64
2023,"""India""","""Russia""",0,"""Clothes""",10
2023,"""India""","""Russia""",1,"""Machinery""",1000


In [22]:
pl.concat((df1, diffdf), how='horizontal_extend')

year,exporter,importer,quantity,item,value
i64,str,str,i64,str,i64
2023,"""India""","""Russia""",0,"""Clothes""",10
2023,"""India""","""Russia""",1,"""Machinery""",1000


In [23]:
df1

year,exporter,importer,quantity
i64,str,str,i64
2023,"""India""","""Russia""",0
2023,"""India""","""Russia""",1


In [27]:
df3 = pl.DataFrame(
    [
        {"year": 2024, "exporter": "India", "importer": "USA", "quantity": 2, "item": "Clothes", "value": 10 },
        {"year": 2024, "exporter": "India", "importer": "USA", "quantity": 3, "item": "Machinery", "value": 1000 }
    ]
)
pl.concat([df1, df3], how='diagonal')

year,exporter,importer,quantity,item,value
i64,str,str,i64,str,i64
2023,"""India""","""Russia""",0,null,null
2023,"""India""","""Russia""",1,null,null
2024,"""India""","""USA""",2,"""Clothes""",10
2024,"""India""","""USA""",3,"""Machinery""",1000


In [29]:
j1 = pl.DataFrame(
    {
        "id": [1,2,3],
        "name": ["Alice", "Bob", "Charlie"]
    }
)
j2 = df2 = pl.DataFrame({
    "id": [2, 3, 4],
    "age": [25, 30, 35]
})
j1

id,name
i64,str
1,"""Alice"""
2,"""Bob"""
3,"""Charlie"""


In [33]:
result = j1.join(j2, on="id", how="inner")
result

id,name,age
i64,str,i64
2,"""Bob""",25
3,"""Charlie""",30


In [35]:
 j1.join(
        j2, 
        on="id",
        how="left",
        coalesce=False #join_nulls
    )
#duplicate columns get suffix

id,name,id_right,age
i64,str,i64,i64
1,"""Alice""",null,null
2,"""Bob""",2,25
3,"""Charlie""",3,30


Note that:
- The order of `df1` is maintained in this left join
- The `null` `value` in the last row if `df1` is not joined to the `null` value in `df2`
- If the join column name(s) are not the same in both `DataFrames` then we specify `left_on` and `right_on` instead of `on`
- as we pass `coalesce=False` (which is the default) we get `id` and `id_right` join columns in the output

If we instead pass `coalesce=True` Polars coalesces the join columns `id` and `id_right` into a single `id` column (this was the default behaviour previously, personally this is what I normally want

prev == join_nulls
Default (nulls_equal=False): Null values in the join keys never match each other. Rows containing null on the join keys are omitted from matches.Enabled (nulls_equal=True): Two null values on the join keys are considered a match (similar to null-safe equality).

In [36]:
 j1.join(
        j2, 
        on="id",
        how="right",
        coalesce=False #how=cross, full
    )

id,name,id_right,age
i64,str,i64,i64
2,"""Bob""",2,25
3,"""Charlie""",3,30
null,null,4,35
